In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.sparse import csr_matrix

def simulate_scatac_seq(
    n_samples=10,
    n_genes=2500,  # In ATAC-seq context, these represent peaks/regions
    n_de_genes=200,
    cells_per_sample=500,
    mean_accessibility=10,  # Lower baseline for ATAC-seq
    desired_sparsity=0.99,  # Higher sparsity for ATAC-seq
    n_recursive_steps=2,
    random_seed=666
):
    """
    Comprehensive scATAC-seq simulation with multimodal accessibility patterns
    
    Parameters:
    -----------
    n_samples: Number of samples (default: 10)
    n_genes: Number of peaks/regions (default: 2500)
    n_de_genes: Number of differentially accessible regions (default: 200)
    cells_per_sample: Number of cells per sample (default: 500)
    mean_accessibility: Average accessibility level (default: 10)
    desired_sparsity: Target sparsity level (default: 0.99)
    n_recursive_steps: Number of recursive scaling steps (default: 2)
    random_seed: Random seed for reproducibility (default: 666)
    """
    
    # Set random seed
    np.random.seed(random_seed)
    
    # Define sample names and group assignments
    sample_names = [f'Sample{i+1}' for i in range(n_samples)]
    sample_to_group = {}
    for i, sample_name in enumerate(sample_names):
        if i < 4:
            sample_to_group[sample_name] = 'Group0'
        else:
            sample_to_group[sample_name] = 'Group1'
    
    # Sample heterogeneity factors (0.9 to 1.1)
    sample_scaling = {name: np.random.uniform(0.9, 1.1) for name in sample_names}
    
    # Define cells per sample with slight variation
    cells_per_sample_dict = {
        name: np.random.poisson(cells_per_sample) for name in sample_names
    }
    
    # Total number of cells
    total_cells = sum(cells_per_sample_dict.values())
    
    # Generate base accessibility levels using Gamma distribution
    # Lower values for ATAC-seq to reflect sparser nature
    lambda_base = np.random.gamma(shape=mean_accessibility/5, scale=5.0, size=n_genes)
    
    # Define accessibility for each group
    lambda_group = {
        'Group0': lambda_base.copy(),
        'Group1': lambda_base.copy()
    }
    
    # Randomly select differentially accessible regions (DARs)
    dar_indices = np.random.choice(n_genes, size=n_de_genes, replace=False)
    dar_indices_open = dar_indices[:100]  # First 100 are more open/accessible
    dar_indices_closed = dar_indices[100:]  # Next 100 are more closed
    
    # Initialize data storage
    all_counts = []
    all_cell_ids = []
    all_sample_ids = []
    all_group_ids = []
    
    # Track DAR scaling info for validation
    scaling_info = {
        'open_factors': [],
        'closed_factors': [],
        'affected_cells': {}
    }
    
    # Simulate data for each sample
    for sample_idx, sample_name in enumerate(sample_names):
        group_id = sample_to_group[sample_name]
        n_cells = cells_per_sample_dict[sample_name]
        sample_scale = sample_scaling[sample_name]
        
        # Apply sample heterogeneity to accessibility
        sample_lambda = lambda_group[group_id] * sample_scale
        
        # Simulate counts using Poisson distribution (appropriate for ATAC-seq)
        counts = np.random.poisson(lam=sample_lambda, size=(n_cells, n_genes))
        
        # Ensure counts are integers and non-negative
        counts = np.maximum(counts, 0).astype(int)
        
        # Apply multimodal scaling for DARs in Group 0
        if group_id == 'Group0':
            # Track which cells are affected
            affected_cells = np.ones(n_cells, dtype=bool)
            scaling_info['affected_cells'][sample_name] = []
            
            # Recursive scaling process
            for step in range(n_recursive_steps):
                # Select 50% of currently active cells
                n_active = np.sum(affected_cells)
                n_to_scale = max(1, int(n_active * 0.5))
                active_indices = np.where(affected_cells)[0]
                selected_indices = np.random.choice(active_indices, size=n_to_scale, replace=False)
                
                # Generate scaling factors for this step
                # For ATAC-seq: open regions get higher accessibility
                open_factors = np.random.uniform(1.1, 8.0, size=len(selected_indices))
                closed_factors = np.random.uniform(0.1, 0.9, size=len(selected_indices))
                
                # Apply scaling to selected cells
                for idx, cell_idx in enumerate(selected_indices):
                    counts[cell_idx, dar_indices_open] = np.floor(
                        counts[cell_idx, dar_indices_open] * open_factors[idx]
                    ).astype(int)
                    counts[cell_idx, dar_indices_closed] = np.floor(
                        counts[cell_idx, dar_indices_closed] * closed_factors[idx]
                    ).astype(int)
                
                # Update affected cells for next iteration
                affected_cells = np.zeros(n_cells, dtype=bool)
                affected_cells[selected_indices] = True
                
                # Store scaling info
                scaling_info['affected_cells'][sample_name].append(selected_indices.tolist())
                scaling_info['open_factors'].extend(open_factors.tolist())
                scaling_info['closed_factors'].extend(closed_factors.tolist())
        
        # Apply expression-dependent dropout mechanism
        # Calculate dropout probabilities
        log_counts = np.log1p(counts)  # log(count + 1)
        max_log_count = np.max(log_counts)
        if max_log_count > 0:
            dropout_prob = (1 - log_counts / max_log_count) * desired_sparsity
        else:
            dropout_prob = np.ones_like(counts) * desired_sparsity
        
        # Apply dropout
        random_values = np.random.rand(*counts.shape)
        counts[random_values < dropout_prob] = 0
        
        # Additional binarization step for ATAC-seq (optional)
        # Uncomment to convert to binary accessibility matrix
        # counts = (counts > 0).astype(int)
        
        # Collect data
        all_counts.append(counts)
        
        # Generate cell IDs
        cell_ids = [f'{sample_name}_Cell{i+1}' for i in range(n_cells)]
        all_cell_ids.extend(cell_ids)
        all_sample_ids.extend([sample_name] * n_cells)
        all_group_ids.extend([group_id] * n_cells)
    
    # Combine all counts
    counts_matrix = np.vstack(all_counts)
    
    # Create peak/region names
    peak_names = [f'Peak_{i+1}' for i in range(n_genes)]
    
    # Mark differentially accessible regions
    peak_info = pd.DataFrame({
        'peak_id': peak_names,
        'is_dar': False,
        'dar_direction': 'none'
    })
    peak_info.loc[dar_indices_open, 'is_dar'] = True
    peak_info.loc[dar_indices_open, 'dar_direction'] = 'open'
    peak_info.loc[dar_indices_closed, 'is_dar'] = True
    peak_info.loc[dar_indices_closed, 'dar_direction'] = 'closed'
    
    # Create cell metadata
    obs_data = pd.DataFrame({
        'cell_id': all_cell_ids,
        'sample': all_sample_ids,
        'group': all_group_ids
    })
    obs_data.index = obs_data['cell_id']
    
    # Create AnnData object
    adata = ad.AnnData(
        X=csr_matrix(counts_matrix),
        obs=obs_data,
        var=peak_info.set_index('peak_id')
    )
    
    # Add simulation parameters to uns
    adata.uns['simulation_params'] = {
        'n_samples': n_samples,
        'n_peaks': n_genes,
        'n_dars': n_de_genes,
        'cells_per_sample': cells_per_sample,
        'mean_accessibility': mean_accessibility,
        'desired_sparsity': desired_sparsity,
        'n_recursive_steps': n_recursive_steps,
        'distribution': 'poisson',
        'random_seed': random_seed,
        'data_type': 'scATAC-seq'
    }
    
    # Calculate and store summary statistics
    actual_sparsity = (counts_matrix == 0).sum() / counts_matrix.size
    
    # Calculate mean accessibility for DARs by group
    group0_mask = np.array(all_group_ids) == 'Group0'
    group1_mask = np.array(all_group_ids) == 'Group1'
    
    stats = {
        'actual_sparsity': actual_sparsity,
        'total_cells': len(all_cell_ids),
        'cells_per_group': {
            'Group0': group0_mask.sum(),
            'Group1': group1_mask.sum()
        },
        'dar_stats': {
            'open_regions': {
                'mean_group0': counts_matrix[group0_mask][:, dar_indices_open].mean(),
                'mean_group1': counts_matrix[group1_mask][:, dar_indices_open].mean(),
                'fold_change': counts_matrix[group0_mask][:, dar_indices_open].mean() / 
                              (counts_matrix[group1_mask][:, dar_indices_open].mean() + 0.001)
            },
            'closed_regions': {
                'mean_group0': counts_matrix[group0_mask][:, dar_indices_closed].mean(),
                'mean_group1': counts_matrix[group1_mask][:, dar_indices_closed].mean(),
                'fold_change': counts_matrix[group0_mask][:, dar_indices_closed].mean() / 
                              (counts_matrix[group1_mask][:, dar_indices_closed].mean() + 0.001)
            }
        },
        'binary_accessibility': {
            'percent_accessible_group0': ((counts_matrix[group0_mask] > 0).sum() / 
                                         counts_matrix[group0_mask].size) * 100,
            'percent_accessible_group1': ((counts_matrix[group1_mask] > 0).sum() / 
                                         counts_matrix[group1_mask].size) * 100
        }
    }
    
    adata.uns['simulation_stats'] = stats
    
    # Print summary
    print("=== scATAC-seq Simulation Summary ===")
    print(f"Total cells: {stats['total_cells']}")
    print(f"Cells in Group0: {stats['cells_per_group']['Group0']}")
    print(f"Cells in Group1: {stats['cells_per_group']['Group1']}")
    print(f"Number of peaks: {n_genes}")
    print(f"Number of DARs: {n_de_genes} (100 open, 100 closed)")
    print(f"Actual sparsity: {actual_sparsity:.4f}")
    print(f"\nDifferentially accessible regions (open):")
    print(f"  Mean in Group0: {stats['dar_stats']['open_regions']['mean_group0']:.4f}")
    print(f"  Mean in Group1: {stats['dar_stats']['open_regions']['mean_group1']:.4f}")
    print(f"  Fold change: {stats['dar_stats']['open_regions']['fold_change']:.4f}")
    print(f"\nDifferentially accessible regions (closed):")
    print(f"  Mean in Group0: {stats['dar_stats']['closed_regions']['mean_group0']:.4f}")
    print(f"  Mean in Group1: {stats['dar_stats']['closed_regions']['mean_group1']:.4f}")
    print(f"  Fold change: {stats['dar_stats']['closed_regions']['fold_change']:.4f}")
    print(f"\nBinary accessibility:")
    print(f"  % accessible in Group0: {stats['binary_accessibility']['percent_accessible_group0']:.2f}%")
    print(f"  % accessible in Group1: {stats['binary_accessibility']['percent_accessible_group1']:.2f}%")
    
    return adata, scaling_info

# Run simulation
if __name__ == "__main__":
    adata_scatac, scaling_info = simulate_scatac_seq()
    
    # Save data
    adata_scatac.write_h5ad('simulated_scatac_data.h5ad')
    print("\nData saved to: simulated_scatac_data.h5ad")